# Qwen2.5-3B tool-call QLoRA on Kaggle T4

Before running:
1. Notebook settings: Accelerator = GPU T4 x2, Internet = On.
2. Add-ons -> Secrets -> add `GITHUB_TOKEN` with a GitHub PAT that can read the private `mas` repo.
3. Run all. The adapter and eval JSON are written under `/kaggle/working`.

3B is used because a 16GB T4 is tight for 7B QLoRA plus evaluation. If you want 7B, change the model id and config, lower `cutoff_len`, and expect a much longer run.

In [ ]:
!nvidia-smi
!df -h /kaggle/working /kaggle/temp 2>/dev/null || df -h /kaggle/working

In [ ]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
repo = "/kaggle/working/mas"

if not os.path.exists(repo):
    subprocess.run(
        ["git", "clone", f"https://{token}@github.com/starthong/mas.git", repo],
        check=True,
    )
else:
    subprocess.run(["git", "-C", repo, "fetch", "origin", "master"], check=True)
    subprocess.run(["git", "-C", repo, "reset", "--hard", "origin/master"], check=True)

In [ ]:
%cd /kaggle/working/mas
!pip install -q -r requirements.txt -r requirements-kaggle.txt
!llamafactory-cli version

In [ ]:
import os
from huggingface_hub import snapshot_download

model_dir = "/kaggle/temp/models/Qwen2.5-3B-Instruct"
os.makedirs(model_dir, exist_ok=True)
snapshot_download("Qwen/Qwen2.5-3B-Instruct", local_dir=model_dir)
print(model_dir)

In [ ]:
%cd /kaggle/working/mas
!python scripts/build_toolcall_dataset.py
!wc -l data/toolcall_sft.jsonl data/test/toolcall_test.jsonl

In [ ]:
%cd /kaggle/working/mas
!CUDA_VISIBLE_DEVICES=0 llamafactory-cli train configs/train_qlora_kaggle.yaml max_samples=50 max_steps=20 output_dir=/kaggle/working/output/smoke
!ls -lh /kaggle/working/output/smoke
from pathlib import Path
assert Path('/kaggle/working/output/smoke/adapter_model.safetensors').exists(), 'smoke training failed: adapter not found'
print('smoke adapter ok')

In [ ]:
%cd /kaggle/working/mas
!CUDA_VISIBLE_DEVICES=0 llamafactory-cli train configs/train_qlora_kaggle.yaml
from pathlib import Path
assert Path('/kaggle/working/output/qwen25-3b-toolcall-lora/adapter_model.safetensors').exists(), 'full training failed: adapter not found'
print('full adapter ok')

In [ ]:
%cd /kaggle/working/mas
!python -m src.eval.run_eval --backend hf --base-model /kaggle/temp/models/Qwen2.5-3B-Instruct --adapter /kaggle/working/output/qwen25-3b-toolcall-lora --data data/test/toolcall_test.jsonl --out results/ft.json --limit 100

In [ ]:
%cd /kaggle/working/mas
!python -m src.eval.run_eval --backend hf --base-model /kaggle/temp/models/Qwen2.5-3B-Instruct --data data/test/toolcall_test.jsonl --out results/base.json --limit 100

Download `output/qwen25-3b-toolcall-lora`, `results/ft.json`, and `results/base.json` from the notebook output before the session ends.

If `llamafactory-cli` is missing, replace the install cell with:
```bash
git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
pip install -e LLaMA-Factory
```